# ICON4Py JW step under `jax.jit` on a Colab TPU, float32

The Jablonowski-Williamson step of `icon4py.model.driver.jax_api` (diffusion and
solve_nonhydro, 5 substeps, no physics, no tracers; 35 levels, dt 300 s), jitted with the
static fields and connectivity tables as arguments.

Runtime → Change runtime type → **TPU**, then *Run all*. If the install cell reports that
JAX was already imported, restart the session and run all again.

Pinned: icon4py `havogt/icon4py@45d0880712100a959ba24c7639c7a3b6440384a9` (branch `jaxad-tpu-colab`), gt4py
`havogt/gt4py@011f8a2fcda5986f58f0e4a5289c857525ecab6b` (branch `jax-diffusion`). The last cell prints a summary to paste back.

In [ ]:
ICON4PY_SHA = "45d0880712100a959ba24c7639c7a3b6440384a9"
GT4PY_SHA = "011f8a2fcda5986f58f0e4a5289c857525ecab6b"
PRECISION = "single"
GRIDS = ["R02B04", "R02B05"]
SMOKE_STEPS = 10
WARMUP_STEPS = 3
TIMED_STEPS = 20
REQUIRE_TPU = True
SAVE_TO_DRIVE = False

In [ ]:
import os
import pathlib
import subprocess
import sys


if "jax" in sys.modules:
    raise RuntimeError("JAX is already imported: restart the session, then run all.")
REPO = "/content/icon4py"
if not pathlib.Path(REPO).is_dir():
    subprocess.run(
        ["git", "clone", "-q", "--filter=blob:none", "https://github.com/havogt/icon4py", REPO],
        check=True,
    )
subprocess.run(["git", "-C", REPO, "checkout", "-q", ICON4PY_SHA], check=True)
packages = [
    "model/common",
    "model/atmosphere/dycore",
    "model/atmosphere/diffusion",
    "model/atmosphere/tracer_advection",
    "model/atmosphere/subgrid_scale_physics/microphysics",
    "model/atmosphere/subgrid_scale_physics/muphys",
    "model/atmosphere/subgrid_scale_physics/physics_driver",
    "model/atmosphere/subgrid_scale_physics/surface",
    "model/atmosphere/subgrid_scale_physics/tmx",
    "model/driver",
]
jax_spec = "jax[tpu]==0.11.2" if REQUIRE_TPU else "jax==0.11.2"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        jax_spec,
        f"gt4py[jax] @ git+https://github.com/havogt/gt4py@{GT4PY_SHA}",
        *[f"{REPO}/{p}" for p in packages],
    ],
    check=True,
)

In [ ]:
os.environ["ICON4PY_FLOAT_PRECISION"] = PRECISION
sys.path.insert(0, f"{REPO}/scripts/colab")
import json
import pathlib
import platform

import jw_run
import numpy as np


jax = jw_run.jax
import gt4py


jax.config.update("jax_compilation_cache_dir", "/content/jax_cache")
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
devices = jax.devices()
print("python", platform.python_version(), "| jax", jax.__version__, "| gt4py", gt4py.__version__)
print("devices", devices)
print("precision", jw_run.type_alias.precision, "wpfloat", np.dtype(jw_run.type_alias.wpfloat).name)
if REQUIRE_TPU and devices[0].platform != "tpu":
    raise RuntimeError(f"Not a TPU runtime: {devices}. Runtime → Change runtime type → TPU.")
assert (
    np.dtype(jw_run.type_alias.wpfloat).name
    == {"single": "float32", "double": "float64"}[PRECISION]
)
head = subprocess.run(
    ["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
assert head.startswith(ICON4PY_SHA), head
summary = {
    "device_kind": devices[0].device_kind,
    "platform": devices[0].platform,
    "num_devices": len(devices),
    "jax": jax.__version__,
    "gt4py": gt4py.__version__,
    "gt4py_commit": GT4PY_SHA,
    "icon4py_commit": head,
    "precision": PRECISION,
    "python": platform.python_version(),
}

In [ ]:
GRID_DIR = pathlib.Path("/content/grids")
grid_files = {name: jw_run.download_grid(name, GRID_DIR) for name in GRIDS}
for name, path in grid_files.items():
    print(name, path, f"{path.stat().st_size / 2**20:.0f} MiB")

## (1) Smoke: R02B04, `SMOKE_STEPS` jitted steps, against the CPU reference

The reference holds the prognostics at a fixed sample of 256 cells / edges (all levels) after
the same steps on CPU, from the same commits, in single and in double precision. The
*double* row is the float32 error against float64 on CPU, the scale to judge the TPU's
difference from the CPU float32 run by. `max_rel` is max |difference| / max |reference| of the sample.

In [ ]:
reference = np.load(f"{REPO}/scripts/colab/jw_reference_R02B04.npz")
ref_meta = json.loads(str(reference["meta"]))
assert ref_meta["config_sha256"] == jw_run.config_sha256(), (
    "jw_config.yml differs from the reference's"
)
assert ref_meta["num_steps"] == SMOKE_STEPS, ref_meta["num_steps"]
print(
    "reference:",
    {k: ref_meta[k] for k in ("icon4py_commit", "gt4py_version", "jax_version", "num_steps")},
)

static, state = jw_run.init(grid_files["R02B04"], pathlib.Path("/content/output"))
report = jw_run.precision_report(static, state, jax.jit(jw_run.jax_api.time_step))
print(json.dumps(report, indent=1))
if PRECISION == "single":
    assert not report["leaves_not_wpfloat"], report["leaves_not_wpfloat"]
    assert report["hlo_f64_first_step"] == 0 and report["hlo_f64_later_steps"] == 0, report

indices = {k.removeprefix("idx_"): reference[k] for k in reference.files if k.startswith("idx_")}
final = jw_run.prognostics(jw_run.run(static, state, SMOKE_STEPS))
diffs = {tag: jw_run.compare(final, reference, indices, tag) for tag in ("single", "double")}
print(
    f"{'field':8} {'finite':6} {'vs CPU f32 max_abs':>18} {'max_rel':>9} {'vs CPU f64 max_abs':>18} {'max_rel':>9}"
)
for name in jw_run.PROGNOSTICS:
    s, d = diffs["single"][name], diffs["double"][name]
    print(
        f"{name:8} {s['finite']!s:6} {s['max_abs']:18.3e} {s['max_rel']:9.2e} {d['max_abs']:18.3e} {d['max_rel']:9.2e}"
    )
finite = all(np.isfinite(v).all() for v in final.values())
assert finite, "non-finite prognostics"
summary["smoke"] = {
    "grid": "R02B04",
    "steps": SMOKE_STEPS,
    "finite": finite,
    "precision_report": report,
    "diffs": diffs,
}
del static, state

## (2) Timing

Ahead-of-time trace + lower and compile of the first step on their own; then the jitted
step's first two calls (the first should load the ahead-of-time compile from the persistent
cache; the second traces again, as `at_initial_timestep` is metadata) and the
median of `TIMED_STEPS` synchronised steps after `WARMUP_STEPS`. R02B05 is skipped with a note
if it does not fit.

In [ ]:
summary["timing"] = {}
for name in GRIDS:
    try:
        static, state = jw_run.init(grid_files[name], pathlib.Path("/content/output"))
        timings, state = jw_run.time_steps(static, state, WARMUP_STEPS, TIMED_STEPS)
        final = jw_run.prognostics(state)
        timings["finite"] = all(np.isfinite(v).all() for v in final.values())
        summary["timing"][name] = timings
        print(name, json.dumps(timings, indent=1))
    except Exception as e:
        summary["timing"][name] = {"error": f"{type(e).__name__}: {str(e)[:500]}"}
        print(name, "FAILED:", summary["timing"][name]["error"])
    finally:
        static = state = None
        jax.clear_caches()

## Comparison: santis, one GH200

Not the same configuration: icon4py-jaxad-33 (santis job 1005673) timed the components alone
at **R02B05, 80 levels, float64**, JAX with the static fields as jit arguments: diffusion
0.843 ms, one solve_nonhydro substep 7.414 ms, so diffusion + 5 substeps ≈ 37.9 ms. The JW
step here has 35 levels and runs in `PRECISION`; no santis number exists for it.

In [ ]:
summary["santis_gh200_reference"] = {
    "source": "icon4py-jaxad-33, santis 1005673, R02B05, 80 levels, float64, jax_args",
    "diffusion_ms": 0.843,
    "solve_nonhydro_substep_ms": 7.414,
    "diffusion_plus_5_substeps_ms": 0.843 + 5 * 7.414,
}
text = json.dumps(summary, indent=1, default=str)
print(text)
if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    out = pathlib.Path("/content/drive/MyDrive/icon4py_jw_tpu.json")
    out.write_text(text)
    print("written", out)